<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_03_01_hpc_distributed_computing_sparkR_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Distributed Computing with {SparkR}


The `SparkR` package provides a light-weight frontend to use Apache Spark from R. It supports distributed data frames and machine learning. In this notebook, we explore how to use SparkR for big data processing and analysis.

## Overview of SparkR

**SparkR** is an R package that provides a lightweight interface for using **Apache Spark** from within the R programming environment. It enables R users to leverage Spark's distributed computing capabilities for large-scale data processing, machine learning, and structured streaming, while working with familiar R syntax and data structures such as data frames.

**Note:** SparkR is **deprecated as of Apache Spark 4.0.0** and will be removed in a future version. Users are encouraged to consider alternative tools or migration paths.

### Key Features of SparkR

1. **Distributed DataFrames (SparkDataFrame)** — SparkR provides `SparkDataFrame`, similar in spirit to R's `data.frame` or `dplyr` but distributed. Common operations include `select`, `filter`, `groupBy` / `summarize`, `arrange`, and column manipulation.

2. **SparkSession** — `sparkR.session()` initializes a `SparkSession`, the entry point to Spark (master URL, memory, and other config).

3. **Data source integration** — Read/write JSON, CSV, Parquet, Hive, and databases via `read.df()` and `write.df()`.

4. **SQL** — Register temporary views and run SQL with `sql()`.

5. **Machine learning (MLlib)** — Formula-based interfaces for classification, regression, clustering, etc.; `write.ml()` / `read.ml()` for models.

6. **User-defined functions** — `dapply()` / `dapplyCollect()`, `gapply()` / `gapplyCollect()`, and `spark.lapply()` for custom R logic on partitions or groups.

7. **Apache Arrow** — Optional faster JVM↔R transfer when `spark.sql.execution.arrow.sparkr.enabled` is enabled.

8. **Structured Streaming** — Spark's streaming API from R.

9. **Data type mapping** — Automatic mapping between R types and Spark SQL types.

10. **Eager execution (REPL)** — With `spark.sql.repl.eagerEval.enabled`, SparkDataFrames can display like local frames in interactive sessions.

### Limitations and Considerations

- **Deprecation:** SparkR is deprecated starting in **Spark 4.0.0**.
- **Name conflicts:** Some SparkR functions (e.g., `filter`, `sample`, `cov`) mask base R or `stats`; use `base::sample` etc. when needed.
- **Arrow:** Not all Spark types are supported under Arrow.
- **Driver memory:** `collect()` and `dapplyCollect()` pull data to the driver; avoid on huge datasets.


## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data files live there. Edit `dataFolder` in the data-setup cell to point at your Parquet and CSV files (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install Apache Spark (Colab)

SparkR loads from `$SPARK_HOME/R/lib`. On Colab, download a **Spark 3.x** distribution (SparkR remains usable there; check Spark release notes for your environment). Adjust `SPARK_VERSION` if needed. **Local use:** set `SPARK_HOME` yourself and skip this cell.


In [ ]:
import os
import tarfile
import urllib.request

SPARK_VERSION = "3.5.4"
HADOOP_PROFILE = "hadoop3"
spark_dir = f"/content/spark-{SPARK_VERSION}-bin-{HADOOP_PROFILE}"
tar_name = f"spark-{SPARK_VERSION}-bin-{HADOOP_PROFILE}.tgz"
url = f"https://archive.apache.org/dist/spark/spark-{SPARK_VERSION}/{tar_name}"

if not os.path.isdir(spark_dir):
    tgz_path = f"/content/{tar_name}"
    if not os.path.isfile(tgz_path):
        print("Downloading Spark...")
        urllib.request.urlretrieve(url, tgz_path)
    print("Extracting...")
    with tarfile.open(tgz_path, "r:gz") as tf:
        tf.extractall("/content")
os.environ["SPARK_HOME"] = spark_dir
os.environ["PATH"] = os.pathsep.join([os.path.join(spark_dir, "bin"), os.environ.get("PATH", "")])
print("SPARK_HOME =", os.environ["SPARK_HOME"])


## Check and Install Required R Packages

Install CRAN packages to Google Drive on Colab. **SparkR** itself is loaded from the Spark installation (`SPARK_HOME/R/lib`), not from CRAN.


### Define required packages


In [ ]:
%%R
packages <- c(
  'tidyverse',
  'ggplot2'
)


### Install missing packages (Google Drive library)


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages


### Check loaded packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


## Basic Data Processing with SparkR

### `SPARK_HOME` and SparkR

The Python cell above sets `SPARK_HOME` in the Colab process. **rpy2** inherits environment variables for child R processes in many setups; if `Sys.getenv("SPARK_HOME")` is empty in R, set it explicitly to the same path as printed above (for example `/content/spark-3.5.4-bin-hadoop3`).


### Starting SparkR in R

Before using SparkR, Apache Spark must be installed. After installation, `SPARK_HOME` should point to the Spark root. SparkR is loaded from `file.path(Sys.getenv("SPARK_HOME"), "R", "lib")`.


In [ ]:
%%R
# Colab: SPARK_HOME is often set by the install cell; local: set your path
sh <- Sys.getenv("SPARK_HOME")
if (nchar(sh) < 1) Sys.setenv(SPARK_HOME = "/content/spark-3.5.4-bin-hadoop3")
library(SparkR, lib.loc = c(file.path(Sys.getenv("SPARK_HOME"), "R", "lib")))
# Avoid dplyr masking SparkR S4 methods in this notebook
if ("package:dplyr" %in% search()) detach("package:dplyr", unload = TRUE)
if ("package:data.table" %in% search()) detach("package:data.table", unload = TRUE)
sparkR.session(
  master = "local[*]",
  appName = "Iris Analysis",
  sparkConfig = list(spark.driver.memory = "2g")
)


### Convert R `data.frame` to `SparkDataFrame`

`as.DataFrame()` converts an R `data.frame` or list into a `SparkDataFrame`.


In [ ]:
%%R
df <- as.DataFrame(iris)
printSchema(df)


### SparkDataFrame operations


In [ ]:
%%R
head(df, 5)


In [ ]:
%%R
filtered_df <- filter(df, df$Sepal_Length > 5.0)
head(filtered_df, 5)


In [ ]:
%%R
grouped_df <- summarize(groupBy(df, df$Species), avg_sepal_length = avg(df$Sepal_Length))
collect(grouped_df)


**OLAP cube** — Aggregations across multiple dimensions with `cube()`.


In [ ]:
%%R
cube_df <- cube(df, df$Species)
cube_df <- summarize(cube_df, avg_sepal_length = avg(df$Sepal_Length))
collect(cube_df)


**Rollup** — Hierarchical aggregations and subtotals with `rollup()`.


In [ ]:
%%R
head(agg(rollup(df, df$Species), avg_sepal_length = avg(df$Sepal_Length)), 5)


### Operating on columns


In [ ]:
%%R
df$Sepal_Area <- df$Sepal_Length * df$Sepal_Width
head(select(df, df$Sepal_Length, df$Sepal_Width, df$Sepal_Area), 5)


### Applying user-defined functions

Use `dapply()` or `gapply()` for custom R functions on partitions or groups.

**dapply** — Apply a function to each partition of a `SparkDataFrame`.


In [ ]:
%%R
data(iris)
iris_df <- as.DataFrame(iris)
printSchema(iris_df)
categorize_sepal_length <- function(df) {
  df$Sepal_Category <- ifelse(df$Sepal_Length > 5.5, "Long", "Short")
  return(df)
}
output_schema <- structType(
  structField("Sepal_Length", "double"),
  structField("Sepal_Width", "double"),
  structField("Petal_Length", "double"),
  structField("Petal_Width", "double"),
  structField("Species", "string"),
  structField("Sepal_Category", "string")
)
df1 <- dapply(iris_df, categorize_sepal_length, output_schema)
result <- select(df1, "Sepal_Length", "Sepal_Category")
head(result, 5)
result_local <- collect(result)
print(result_local)
library(ggplot2)
ggplot(result_local, aes(x = Sepal_Category)) +
  geom_bar(fill = "steelblue") +
  labs(title = "Count of Sepal Length Categories", x = "Sepal Category", y = "Count") +
  theme_minimal()


**gapply** — Group by column(s), apply an R function to each group. Example: Old Faithful — group by `waiting`, take max `eruptions` per group.


In [ ]:
%%R
sparkR.session(
  master = "local[*]",
  appName = "Old Faithful Analysis",
  sparkConfig = list(spark.driver.memory = "2g")
)
data(faithful)
faithful_df <- as.DataFrame(faithful)
printSchema(faithful_df)
schema <- structType(
  structField("waiting", "double"),
  structField("max_eruption", "double")
)
result <- gapply(
  faithful_df,
  "waiting",
  function(key, x) {
    y <- data.frame(key, max(x$eruptions))
    return(y)
  },
  schema
)
sorted_result <- arrange(result, desc(result$max_eruption))
top_six <- head(collect(sorted_result), 6)
print(top_six)


## Large Data Processing with SparkR

NYC Yellow Taxi data (Parquet) and taxi zone lookup (CSV), following the Quarto tutorial.

### Data paths (Colab vs local)


In [ ]:
%%R
# Colab: uncomment and set your Drive folder
# dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
# dataFolder <- paste0(sub("/$", "", dataFolder), "/")

# Local / repo (edit if needed)
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
trip_data_path <- paste0(dataFolder, "yellow_tripdata_2023-01.parquet")
zone_lookup_path <- paste0(dataFolder, "taxi_zone_lookup.csv")
trips <- read.df(trip_data_path, source = "parquet")
zones <- read.df(zone_lookup_path, source = "csv", header = "true", inferSchema = "true")


### Schemas


In [ ]:
%%R
printSchema(trips)
printSchema(zones)


### Exploring the data

Register temporary views for SQL.


In [ ]:
%%R
createOrReplaceTempView(trips, "trips")
createOrReplaceTempView(zones, "zones")
trip_count <- sql("SELECT COUNT(*) AS total_trips FROM trips")
collect(trip_count)
head(trips, 5)


### Summary statistics


In [ ]:
%%R
summary_trips <- summarize(trips,
  avg_distance = mean(trips$trip_distance),
  avg_fare = mean(trips$total_amount),
  max_distance = max(trips$trip_distance)
)
collect(summary_trips)


### Joining datasets


In [ ]:
%%R
joined_trips <- sql("
  SELECT t.*, z.Borough AS pickup_borough, z.Zone AS pickup_zone
  FROM trips t
  JOIN zones z ON t.PULocationID = z.LocationID
")
createOrReplaceTempView(joined_trips, "joined_trips")


## Data analysis examples


### Average trip distance by pickup borough


In [ ]:
%%R
avg_distance_borough <- sql("
  SELECT pickup_borough, AVG(trip_distance) AS avg_distance
  FROM joined_trips
  GROUP BY pickup_borough
  ORDER BY avg_distance DESC
")
collect(avg_distance_borough)


### Total fare amount by payment type


In [ ]:
%%R
total_fare_payment <- sql("
  SELECT payment_type, SUM(total_amount) AS total_fare
  FROM trips
  GROUP BY payment_type
  ORDER BY total_fare DESC
")
collect(total_fare_payment)


### Average fare by pickup zone


In [ ]:
%%R
avg_fare_by_zone <- sql("
  SELECT pickup_zone, AVG(total_amount) AS avg_fare, COUNT(*) AS trip_count
  FROM joined_trips
  WHERE total_amount > 0 AND trip_distance > 0
  GROUP BY pickup_zone
  HAVING trip_count > 100
  ORDER BY avg_fare DESC
")
collect(avg_fare_by_zone)


### Trip duration analysis


In [ ]:
%%R
trips_with_duration <- sql("
  SELECT *,
    (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60 AS duration_minutes
  FROM trips
  WHERE tpep_dropoff_datetime > tpep_pickup_datetime
")
createOrReplaceTempView(trips_with_duration, "trips_with_duration")
avg_duration <- sql("
  SELECT passenger_count, AVG(duration_minutes) AS avg_duration
  FROM trips_with_duration
  GROUP BY passenger_count
  ORDER BY passenger_count
")
collect(avg_duration)


### Peak hours for pickups


In [ ]:
%%R
peak_hours <- sql("
  SELECT HOUR(tpep_pickup_datetime) AS pickup_hour, COUNT(*) AS trip_count
  FROM trips
  GROUP BY pickup_hour
  ORDER BY trip_count DESC
")
collect(peak_hours)


### Popular pickup boroughs (plot)


In [ ]:
%%R
popular_boroughs <- sql("
  SELECT pickup_borough, COUNT(*) AS trip_count
  FROM joined_trips
  GROUP BY pickup_borough
  ORDER BY trip_count DESC
")
popular_boroughs_local <- collect(popular_boroughs)
library(ggplot2)
ggplot(popular_boroughs_local, aes(x = reorder(pickup_borough, -trip_count), y = trip_count)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  labs(title = "Most Popular Pickup Boroughs (Jan 2023)", x = "Borough", y = "Trip Count") +
  theme_minimal()


### Correlation analysis

SparkR `corr()` computes Pearson correlation in a distributed way on selected columns.


In [ ]:
%%R
trip_data <- select(trips, "trip_distance", "fare_amount", "tip_amount")
trip_data <- na.omit(trip_data)
showDF(trip_data, numRows = 5)
cat("Distributed Pearson correlations:\n")
cor_trip_fare <- corr(trip_data, "trip_distance", "fare_amount", method = "pearson")
cor_trip_tip <- corr(trip_data, "trip_distance", "tip_amount", method = "pearson")
cor_fare_tip <- corr(trip_data, "fare_amount", "tip_amount", method = "pearson")
cat("Distributed Pearson correlation matrix:\n")
cor_matrix <- matrix(c(1, cor_trip_fare, cor_trip_tip,
                      cor_trip_fare, 1, cor_fare_tip,
                      cor_trip_tip, cor_fare_tip, 1),
                    nrow = 3, byrow = TRUE)
print(cor_matrix)


### Simple linear regression (`spark.lm`)

Predict `fare_amount` from `trip_distance`.


In [ ]:
%%R
simple_model <- spark.lm(trip_data, fare_amount ~ trip_distance,
                         regParam = 0.01,
                         maxIter = 100L)
summary_simple <- summary(simple_model)
cat("Simple linear regression summary (fare_amount ~ trip_distance):\n")
print(summary_simple)


### Multi-variable linear regression


In [ ]:
%%R
multi_model <- spark.lm(trip_data, fare_amount ~ trip_distance + tip_amount,
                        regParam = 0.01,
                        maxIter = 100L)
summary_multi <- summary(multi_model)
cat("Multiple linear regression summary (fare_amount ~ trip_distance + tip_amount):\n")
print(summary_multi)


### Stop Spark session (optional)

Release Spark resources when finished.


In [ ]:
%%R
sparkR.stop()


## Summary and conclusion

This notebook covered setting up SparkR, basic `SparkDataFrame` operations, UDFs with `dapply` and `gapply`, NYC taxi analysis with SQL, correlations, and `spark.lm` regression.


## References

1. [Mastering Spark with R](https://therinspark.com/)
2. [MIT OCW — Distributed Systems](https://ocw.mit.edu/courses/6-824-distributed-computer-systems-engineering-spring-2006/)
3. [Use SparkR (Microsoft Fabric)](https://learn.microsoft.com/en-us/fabric/data-science/r-use-sparkr)
4. [Databricks — Getting started with Apache Spark](https://www.databricks.com/spark/getting-started-with-apache-spark)
